In [ ]:
from IPython import display

In [ ]:
try:
    import nbformat
except:
    !pip install nbformat
display.clear_output()

In [ ]:
%run ../model/train.ipynb

In [ ]:
!jupyter nbconvert  ../model/train.ipynb --to html 

# Import lib


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve
import joblib

data = np.load("../../exps/trainbase_18102025/data.npz", allow_pickle=True)

In [ ]:
with np.load('../../exps/trainbase_18102025/data.npz') as z:
    print(z.files)          # shows the list of keys present

In [ ]:
# L?y d? li?u train v? test
df_train = pd.DataFrame(data = data['train_data'], columns= data['train_columns'])
df_test = pd.DataFrame(data['test_data'], columns=df_train.drop(columns=['Output']).columns)


In [ ]:
df_train

In [ ]:
df_test

In [ ]:
X_train = df_train.drop(columns=['Output', 'PassengerId'], axis=1)
y_train = df_train['Output']

In [ ]:
X_test = df_test

# Logistic Regression


In [ ]:
# T?o pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=42, max_iter=1000))
])

# Thi?t l?p hyperparameters
param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l1', 'l2'],
    'logreg__solver': ['liblinear']
}

# S? d?ng Stratified K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search v?i multiple scoring metrics
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=cv, 
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='accuracy',  # Ch?n metric ?? ch?n best model
    n_jobs=-1,
    verbose=1
)

# Train model
print("\nB?t ??u training v?i GridSearch...")
grid_search.fit(X_train, y_train)

# K?t qu?
print("\nBest parameters:", grid_search.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search.best_score_))

# L?y t?t c? k?t qu? cross-validation
cv_results = grid_search.cv_results_
print("\nCross-validation results:")
for i in range(len(cv_results['params'])):
    print(f"Params: {cv_results['params'][i]}")
    print(f"  Accuracy: {cv_results['mean_test_accuracy'][i]:.4f} (+/- {cv_results['std_test_accuracy'][i]:.4f})")
    print(f"  F1: {cv_results['mean_test_f1'][i]:.4f} (+/- {cv_results['std_test_f1'][i]:.4f})")
    print(f"  ROC AUC: {cv_results['mean_test_roc_auc'][i]:.4f} (+/- {cv_results['std_test_roc_auc'][i]:.4f})")

In [ ]:
# Model t?t nh?t
best_model_lr = grid_search.best_estimator_

# ??nh gi? tr?n training set
y_pred_train = best_model_lr.predict(X_train)
y_pred_proba_train = best_model_lr.predict_proba(X_train)[:, 1]  # X?c su?t l?p 1

# T?nh c?c metrics
train_accuracy = best_model_lr.score(X_train, y_train)
train_f1 = f1_score(y_train, y_pred_train)
train_roc_auc = roc_auc_score(y_train, y_pred_proba_train)

print(f"\n=== K?T QU? TR?N TRAINING SET ===")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"F1 Score: {train_f1:.4f}")
print(f"ROC AUC: {train_roc_auc:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_train, y_pred_train))

## Dump m? h?nh logistic regression


In [ ]:
import joblib

# L?u model
joblib.dump(best_model_lr, '../../exps/trainbase_18102025/logistic_regression_model.pkl')
print("Model ?? ???c l?u b?ng joblib")

# Tham s?
feature_names = X_train.columns.tolist()
scaler = best_model_lr.named_steps['scaler']

# L?u nhi?u objects c?ng l?c
joblib.dump({
    'model': best_model_lr,
    'feature_names': feature_names,  # N?u c?
    'scaler': scaler,  # N?u c?
    'training_date': '2024-10-18'
}, '../../exps/trainbase_18102025/model_package.pkl')

# Load l?i model
loaded_model = joblib.load('../../exps/trainbase_18102025/logistic_regression_model.pkl')

# Random forest


In [ ]:
# T?o pipeline cho Random Forest
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),  # V?n c? th? chu?n h?a cho RF (kh?ng b?t bu?c)
    ('rf', RandomForestClassifier(random_state=42))
])

# Thi?t l?p hyperparameters cho Random Forest
param_grid_rf = {
    'rf__n_estimators': [50, 100, 200],           # S? c?y trong r?ng
    'rf__max_depth': [3, 5, 7, None],             # ?? s?u t?i ?a c?a c?y
    'rf__min_samples_split': [2, 5, 10],          # S? m?u t?i thi?u ?? split
    'rf__min_samples_leaf': [1, 2, 4],            # S? m?u t?i thi?u ? l?
    'rf__max_features': ['sqrt', 'log2', None]    # S? features x?t t?i m?i split
}

# S? d?ng Stratified K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search v?i multiple scoring metrics
grid_search_rf = GridSearchCV(
    pipeline_rf, 
    param_grid_rf, 
    cv=cv, 
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='accuracy',  # Ch?n metric ?? ch?n best model
    n_jobs=-1,
    verbose=1
)

# Train model
print("\nB?t ??u training Random Forest v?i GridSearch...")
grid_search_rf.fit(X_train, y_train)

# K?t qu?
print("\nBest parameters:", grid_search_rf.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search_rf.best_score_))

# L?y t?t c? k?t qu? cross-validation
cv_results = grid_search_rf.cv_results_
print("\nCross-validation results (top 5 combinations):")
# Ch? hi?n th? 5 combination t?t nh?t
best_indices = np.argsort(cv_results['mean_test_accuracy'])[-5:][::-1]

for i in best_indices:
    print(f"\nParams: {cv_results['params'][i]}")
    print(f"  Accuracy: {cv_results['mean_test_accuracy'][i]:.4f} (+/- {cv_results['std_test_accuracy'][i]:.4f})")
    print(f"  F1: {cv_results['mean_test_f1'][i]:.4f} (+/- {cv_results['std_test_f1'][i]:.4f})")
    print(f"  ROC AUC: {cv_results['mean_test_roc_auc'][i]:.4f} (+/- {cv_results['std_test_roc_auc'][i]:.4f})")

# Model t?t nh?t
best_model_rf = grid_search_rf.best_estimator_

# ??nh gi? tr?n training set
y_pred_train = best_model_rf.predict(X_train)
y_pred_proba_train = best_model_rf.predict_proba(X_train)[:, 1]  # X?c su?t l?p 1

# T?nh c?c metrics
train_accuracy = best_model_rf.score(X_train, y_train)
train_f1 = f1_score(y_train, y_pred_train)
train_roc_auc = roc_auc_score(y_train, y_pred_proba_train)

In [ ]:
print(f"\n=== K?T QU? TR?N TRAINING SET ===")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"F1 Score: {train_f1:.4f}")
print(f"ROC AUC: {train_roc_auc:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_train, y_pred_train))

# Dump m? h?nh Random Forest


In [ ]:
import joblib

# L?u model
joblib.dump(best_model_rf, '../../exps/trainbase_18102025/random_forest_model.pkl')
print("Model ?? ???c l?u b?ng joblib")

# Tham s?
feature_names = X_train.columns.tolist()
scaler = best_model_rf.named_steps['scaler']

# L?u nhi?u objects c?ng l?c
joblib.dump({
    'model': best_model_rf,
    'feature_names': feature_names,  # N?u c?
    'scaler': scaler,  # N?u c?
    'training_date': '2024-10-18'
}, '../../exps/trainbase_18102025/model_package.pkl')

# Load l?i model
loaded_model = joblib.load('../../exps/trainbase_18102025/random_forest_model.pkl')

# XGBoost


In [ ]:
# T?o pipeline cho XGBoost
pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),  # XGBoost th??ng kh?ng c?n scaling nh?ng v?n c? th? d?ng
    ('xgb', XGBClassifier(random_state=42, eval_metric='logloss'))
])

# Thi?t l?p hyperparameters cho XGBoost
param_grid_xgb = {
    'xgb__n_estimators': [50, 100, 200],           # S? c?y (boosting rounds)
    'xgb__max_depth': [3, 5, 7],                   # ?? s?u t?i ?a c?a c?y
    'xgb__learning_rate': [0.01, 0.1, 0.2],       # T?c ?? h?c
    'xgb__subsample': [0.8, 0.9, 1.0],            # T? l? m?u cho m?i c?y
    'xgb__colsample_bytree': [0.8, 0.9, 1.0],     # T? l? features cho m?i c?y
    'xgb__reg_alpha': [0, 0.1, 1],                # L1 regularization
    'xgb__reg_lambda': [1, 10, 100]               # L2 regularization
}

# S? d?ng Stratified K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search v?i multiple scoring metrics
grid_search_xgb = GridSearchCV(
    pipeline_xgb, 
    param_grid_xgb, 
    cv=cv, 
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
    refit='accuracy',  # Ch?n metric ?? ch?n best model
    n_jobs=-1,
    verbose=1
)

# Train model
print("\nB?t ??u training XGBoost v?i GridSearch...")
grid_search_xgb.fit(X_train, y_train)

# K?t qu?
print("\nBest parameters:", grid_search_xgb.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search_xgb.best_score_))

# L?y t?t c? k?t qu? cross-validation
cv_results = grid_search_xgb.cv_results_
print("\nCross-validation results (top 5 combinations):")
# Ch? hi?n th? 5 combination t?t nh?t
best_indices = np.argsort(cv_results['mean_test_accuracy'])[-5:][::-1]

for i in best_indices:
    print(f"\nParams: {cv_results['params'][i]}")
    print(f"  Accuracy: {cv_results['mean_test_accuracy'][i]:.4f} (+/- {cv_results['std_test_accuracy'][i]:.4f})")
    print(f"  F1: {cv_results['mean_test_f1'][i]:.4f} (+/- {cv_results['std_test_f1'][i]:.4f})")
    print(f"  ROC AUC: {cv_results['mean_test_roc_auc'][i]:.4f} (+/- {cv_results['std_test_roc_auc'][i]:.4f})")

# Model t?t nh?t
best_model_xgb = grid_search_xgb.best_estimator_

# ??nh gi? tr?n training set
y_pred_train = best_model_xgb.predict(X_train)
y_pred_proba_train = best_model_xgb.predict_proba(X_train)[:, 1]  # X?c su?t l?p 1

# T?nh c?c metrics
train_accuracy = best_model_xgb.score(X_train, y_train)
train_f1 = f1_score(y_train, y_pred_train)
train_roc_auc = roc_auc_score(y_train, y_pred_proba_train)

In [ ]:

print(f"\n=== K?T QU? TR?N TRAINING SET ===")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"F1 Score: {train_f1:.4f}")
print(f"ROC AUC: {train_roc_auc:.4f}")

# print("\nClassification Report:")
# print(classification_report(y_train, y_pred_train))

# Ch?n ra m? h?nh t?i ?u


In [ ]:
# S? d?ng best model t? GridSearch
best_model = best_model_xgb

# Ki?m tra model ?? ???c train ch?a
print("Model ?? ???c train:", hasattr(best_model.named_steps['xgb'], 'feature_importances_'))

# D? ?o?n tr?n test set
test_preds = best_model.predict(X_test)
test_probas = best_model.predict_proba(X_test)[:, 1]  # N?u c?n x?c su?t

print(f"S? l??ng predictions: {len(test_preds)}")
print(f"Ph?n b? predictions: {pd.Series(test_preds).value_counts().sort_index()}")

test_data_orig = pd.read_csv("../../../data/test.csv")

# T?o submission
submission = pd.DataFrame({
    "PassengerId": test_data_orig["PassengerId"],
    "Survived": test_preds
})

# L?u submission
submission.to_csv("../../exps/trainbase_18102025/submission_xgboost.csv", index=False)
print("Submission file ?? ???c l?u!")

# Ki?m tra submission
print("\n=== SUBMISSION INFO ===")
print(f"Shape: {submission.shape}")
print(f"PassengerId range: {submission['PassengerId'].min()} - {submission['PassengerId'].max()}")
print(f"Survived distribution:\n{submission['Survived'].value_counts().sort_index()}")

# Dump m? h?nh XGBoost


In [ ]:
import joblib

# L?u model
joblib.dump(best_model_xgb, '../../exps/trainbase_18102025/xgboost_model.pkl')
print("Model ?? ???c l?u b?ng joblib")

# Tham s?
feature_names = X_train.columns.tolist()
scaler = best_model_xgb.named_steps['scaler']

# L?u nhi?u objects c?ng l?c
joblib.dump({
    'model': best_model_xgb,
    'feature_names': feature_names,  # N?u c?
    'scaler': scaler,  # N?u c?
    'training_date': '2024-10-18'
}, '../../exps/trainbase_18102025/model_package.pkl')

# Load l?i model
loaded_model = joblib.load('../../exps/trainbase_18102025/xgboost_model.pkl')